# Customer Churn Storyboard – Week 4 EDA

**Objective:** Explore customer attrition in the IBM Telco Customer Churn dataset and answer three business questions:
1. What is the overall customer churn rate?
2. How does contract type relate to churn?
3. How does payment method relate to churn?

**Tools:** pandas, seaborn, matplotlib, Jupyter Notebook.

> Run the notebook from top to bottom. The dataset is downloaded directly from the public IBM sample repository. Charts are saved into the `charts/` folder.

## 1. Import libraries

In [ ]:
import os
import warnings
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)

print("Libraries imported successfully.")

## 2. Download and load the dataset

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect the data

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 4. Clean the data

In [ ]:
df = df.drop_duplicates().copy()

# Remove leading/trailing spaces from text columns.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# TotalCharges is stored as text in this dataset; convert it to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Cleaned dataset shape:", df.shape)
print("Remaining missing values:", int(df.isnull().sum().sum()))
display(df.head())

## 5. Business Question 1 – Overall attrition baseline

In [ ]:
churn_counts = df["Churn"].value_counts()
total_customers = len(df)
churned_customers = int((df["Churn"] == "Yes").sum())
retained_customers = int((df["Churn"] == "No").sum())
churn_rate = churned_customers / total_customers * 100

print(f"Total customers: {total_customers:,}")
print(f"Customers churned: {churned_customers:,}")
print(f"Customers retained: {retained_customers:,}")
print(f"Overall churn rate: {churn_rate:.2f}%")

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="Churn", hue="Churn",
                   order=["No", "Yes"], palette="Set2", legend=False)
ax.set_title("Overall Customer Churn Distribution")
ax.set_xlabel("Customer churn")
ax.set_ylabel("Number of customers")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.tight_layout()
plt.savefig("charts/overall_churn.png", dpi=300, bbox_inches="tight")
plt.show()

**Business interpretation:** The chart compares customers who stayed with the company with those who left. The calculated churn percentage is the baseline attrition rate for this dataset. The business can use this baseline to monitor whether future retention initiatives reduce customer losses.

## 6. Business Question 2 – Contract impact

In [ ]:
contract_summary = (
    df.groupby("Contract")
      .agg(customers=("Churn", "size"),
           churned=("Churn", lambda s: (s == "Yes").sum()),
           churn_rate=("Churn", lambda s: (s == "Yes").mean() * 100))
      .sort_values("churn_rate", ascending=False)
)
display(contract_summary.round(2))

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=contract_summary.reset_index(),
                 x="Contract", y="churn_rate", hue="Contract",
                 palette="Blues_r", legend=False)
ax.set_title("Churn Rate by Contract Type")
ax.set_xlabel("Contract type")
ax.set_ylabel("Customers churned (%)")
ax.set_ylim(0, 100)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")
plt.tight_layout()
plt.savefig("charts/contract_churn.png", dpi=300, bbox_inches="tight")
plt.show()

**Business interpretation:** Compare the churn percentages across month-to-month, one-year, and two-year contracts. A higher rate for month-to-month customers would indicate that this segment is more likely to leave. The company could test loyalty benefits or incentives for longer-term plans; this association does not prove that contract length causes churn.

## 7. Business Question 3 – Billing and payment factors

In [ ]:
payment_summary = (
    df.groupby("PaymentMethod")
      .agg(customers=("Churn", "size"),
           churned=("Churn", lambda s: (s == "Yes").sum()),
           churn_rate=("Churn", lambda s: (s == "Yes").mean() * 100))
      .sort_values("churn_rate", ascending=False)
)
display(payment_summary.round(2))

In [ ]:
plt.figure(figsize=(11, 6))
plot_data = payment_summary.reset_index()
ax = sns.barplot(data=plot_data, x="PaymentMethod", y="churn_rate",
                 hue="PaymentMethod", palette="Set2", legend=False)
ax.set_title("Churn Rate by Payment Method")
ax.set_xlabel("Payment method")
ax.set_ylabel("Customers churned (%)")
ax.set_ylim(0, 100)
plt.xticks(rotation=15, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")
plt.tight_layout()
plt.savefig("charts/payment_churn.png", dpi=300, bbox_inches="tight")
plt.show()

**Business interpretation:** This chart compares the percentage of customers who churn within each payment-method group. If electronic-check customers show a higher churn percentage, the company should investigate whether payment friction, billing experience, or customer mix is associated with that pattern. Correlation does not establish causation.

## 8. Final summary and recommendations

In [ ]:
highest_contract = contract_summary["churn_rate"].idxmax()
highest_contract_rate = contract_summary["churn_rate"].max()
highest_payment = payment_summary["churn_rate"].idxmax()
highest_payment_rate = payment_summary["churn_rate"].max()

print("CUSTOMER CHURN – FINAL BUSINESS SUMMARY")
print("-" * 45)
print(f"Overall churn rate: {churn_rate:.2f}%")
print(f"Contract group with highest churn: {highest_contract} ({highest_contract_rate:.2f}%)")
print(f"Payment method with highest churn: {highest_payment} ({highest_payment_rate:.2f}%)")

print("\\nRecommendations:")
print("1. Monitor churn regularly against the overall baseline.")
print("2. Investigate why customers on high-churn contract types leave.")
print("3. Review the billing experience for payment groups with higher churn.")
print("4. Test targeted retention offers and measure their impact.")

## 9. Export analysis tables

The following files are optional supporting outputs that can be committed to GitHub.

In [ ]:
contract_summary.to_csv("contract_churn_summary.csv")
payment_summary.to_csv("payment_churn_summary.csv")
print("Summary tables and charts have been saved.")